# Lab 2 - Optimization

In [ ]:
# Google Colab
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/tomaztc/Tomav2.git"
    REPO_DIR = "/content/Tomav2/"

    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}

    os.chdir(REPO_DIR)

In [ ]:
# Imports
from importlib.util import find_spec
if find_spec("matplotlib") is None or find_spec("scipy") is None or find_spec("numpy") is None \
    or find_spec("pandas") is None or find_spec("seaborn") is None or find_spec("pymoo") is None:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "matplotlib", "scipy", "numpy", "pandas", "seaborn", "pymoo"])

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import seaborn as sns
import scipy
import time

from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize
from pymoo.core.problem import ElementwiseProblem

from designTool.constants import gravity
from designTool.analyze import analyze
from designTool.geometry import geometry
from designTool.standard_airplane import standard_airplane
from designTool.plots import plot_geometry
from designTool.auxiliary import atmosphere
from designTool.aerodynamics import aerodynamics

# Config
np.random.seed(1234)
pd.options.display.float_format = "{:,.2f}".format
warnings.simplefilter("error")
sns.set_theme(style="whitegrid", context="notebook")

# Constantes
TOMAV_KWARGS = ["delta_xr_w", "x_tank_c_w"]
CONDICAO_SEGUNDO_SEGMENTO = {
    "Mach": 0.30,
    "altitude": 0,
    "n_engines_failed": 1,
    "highlift_config": "takeoff",
    "lg_down": 1,
    "h_ground": 10.668,
}

In [ ]:
# Funções
def aerodynamics_condicao_e_CL(airplane: dict, condicao: dict, CL: float) -> tuple[float, float, dict]:
    CD, CLmax, dragDict = aerodynamics(airplane, condicao["Mach"], condicao["altitude"], CL,
                                       n_engines_failed=condicao["n_engines_failed"],
                                       highlift_config=condicao["highlift_config"], lg_down=condicao["lg_down"],
                                       h_ground=condicao["h_ground"])
    return CD, CLmax, dragDict

def criar_airplane_parametros(airplane_name: str, parametros: dict) -> dict:
    airplane = standard_airplane(airplane_name, **{parametro: valor for parametro, valor in parametros.items() if airplane_name=="Tomav" and parametro in TOMAV_KWARGS})

    for parametro, valor in parametros.items():
        if airplane_name=="Tomav" and parametro in TOMAV_KWARGS:
            pass
        airplane["inputs"][parametro] = valor

    return airplane

def calcular_dados_otimizacao(airplane_name: str, x_norm: np.ndarray, lb: np.ndarray, ub: np.ndarray, nomes_parametros: list) -> dict | None:
    key = tuple(np.round(x_norm, 11))
    if key in cache:
        return cache[key]
    
    x = desnormalizar(x_norm, lb, ub)
    parametros = {parametro: valor for parametro, valor in zip(nomes_parametros, x)}
    airplane = criar_airplane_parametros(airplane_name, parametros)

    try:
        analyze(airplane)
    except:
        return None

    W0 = airplane["thrust_matching"]["W0"]
    Wf = airplane["thrust_matching"]["W_fuel"]
    T0 = airplane["thrust_matching"]["T0"]
    sigma = atmosphere(airplane["inputs"]["altitude_takeoff"], airplane["inputs"]["deltaISA_takeoff"])["density"] / 1.225
    _, CLmax_TO, _ = aerodynamics_condicao_e_CL(airplane, CONDICAO_SEGUNDO_SEGMENTO, 0)
    d_TO = 0.2387 / (sigma * CLmax_TO * airplane["inputs"]["S_w"]) * W0**2 / T0

    dados = {
        # Saídas
        "W0": W0,
        "Wf": Wf,
        # Restrições
        "deltaS_wlan": airplane["thrust_matching"]["deltaS_wlan"], # 1
        "SM_fwd": airplane["balance"]["SM_fwd"], # 2
        "SM_aft": airplane["balance"]["SM_aft"], # 3
        "frac_nlg_aft": airplane["landing_gear"]["frac_nlg_aft"], # 4
        "frac_nlg_fwd": airplane["landing_gear"]["frac_nlg_fwd"], # 5
        "alpha_tipback": airplane["landing_gear"]["alpha_tipback"], # 6
        "alpha_tailstrike": airplane["landing_gear"]["alpha_tailstrike"], # 7
        "phi_overturn": airplane["landing_gear"]["phi_overturn"], # 8
        # Restrição 9 já está definida no input b_tank_b_w = 0.95
        "b_w": airplane["geometry"]["b_w"], # 10
        # Restrição 11 já está definida no input y_mlg = 5.5
        "tail_height": airplane["geometry"]["zt_v"] - airplane["inputs"]["z_lg"], # 12
        # Restrições adicionais:
        "CLv": airplane["balance"]["CLv"], # 13
        "tank_excess": airplane["balance"]["tank_excess"], # 14
        "mlg_xcg_margin": airplane["inputs"]["x_mlg"] - airplane["balance"]["xcg_aft"], # 15
        "d_TO": d_TO, # 16
    }
    cache[key] = dados
    return dados

def normalizar(x: np.ndarray, lb: np.ndarray, ub: np.ndarray) -> np.ndarray:
    return (x - lb) / (ub - lb)

def desnormalizar(x_norm: np.ndarray, lb: np.ndarray, ub: np.ndarray) -> np.ndarray:
    x = lb + x_norm * (ub - lb)
    return np.clip(x, lb, ub)

def funcao_objetivo(airplane_name: str, x_norm: np.ndarray, lb: np.ndarray, ub: np.ndarray, nomes_parametros: list, W0_inicial: float) -> float:
    dados = calcular_dados_otimizacao(airplane_name, x_norm, lb, ub, nomes_parametros)
    x = desnormalizar(x_norm, lb, ub)
    parametros_hist.append(x)
    W0_hist.append(dados["W0"] / gravity if dados is not None else np.nan)
    if dados is None:
        return 1e10
    else:
        return dados["W0"] / W0_inicial

def confun(x_norm: np.ndarray, airplane_name: str, lb: np.ndarray, ub: np.ndarray, nomes_parametros: list, restricoes_limites: dict) -> np.ndarray:
    # ATTENTION: It is really important to normalize input variables and constraints. For example, the aft static margin constraint can be defined as: (SM_aft/0.05 −1) ≥ 0.
    dados = calcular_dados_otimizacao(airplane_name, x_norm, lb, ub, nomes_parametros)
    g = []
    for nome_restricao, (limite_inferior, limite_superior) in restricoes_limites.items():
        if limite_inferior is not None:
            if dados is None:
                g.append(-1000)
            elif limite_inferior == 0:
                g.append(dados[nome_restricao])
            else:
                g.append((dados[nome_restricao] - limite_inferior) / abs(limite_inferior))
        if limite_superior is not None:
            if dados is None:
                g.append(-1000)
            elif limite_superior == 0:
                g.append(-dados[nome_restricao])
            else:
                g.append((limite_superior - dados[nome_restricao]) / abs(limite_superior))

    g_hist.append(g)      
    return np.array(g)

def otimizar_aviao(airplane_name: str, parametros_limites: dict, parametros_iniciais: np.ndarray, restricoes_limites: dict) -> tuple:
    lb = np.array([limite[0] for limite in parametros_limites.values()])
    ub = np.array([limite[1] for limite in parametros_limites.values()])
    nomes_parametros = list(parametros_limites.keys())
    x_norm_inicial = normalizar(parametros_iniciais, lb, ub)
    dados_inicial = calcular_dados_otimizacao(airplane_name, x_norm_inicial, lb, ub, nomes_parametros)
    if dados_inicial is None:
        raise ValueError
    W0_inicial = dados_inicial["W0"]
    start = time.perf_counter()
    result = scipy.optimize.minimize(
        fun=lambda x_norm: funcao_objetivo(airplane_name, x_norm, lb, ub, nomes_parametros, W0_inicial),
        x0=normalizar(parametros_iniciais, lb, ub),
        method="SLSQP",
        bounds=[(0.0, 1.0)] * len(parametros_limites),
        constraints=[{
            'type': 'ineq',
            'fun': lambda x_norm: confun(x_norm, airplane_name, lb, ub, nomes_parametros, restricoes_limites)
        }],
        options={"ftol": 1e-7, "disp": True, "maxiter": 500, "eps": 1e-3}, 
    )
    elapsed = time.perf_counter() - start

    dados_opt = calcular_dados_otimizacao(airplane_name, result.x, lb, ub, nomes_parametros)
    if dados_opt is None:
        raise ValueError
    x = desnormalizar(result.x, lb, ub)
    parametros = {parametro: valor for parametro, valor in zip(nomes_parametros, x)}
    airplane_opt = criar_airplane_parametros(airplane_name, parametros)
    analyze(airplane_opt)
    return result, elapsed, parametros, airplane_opt, dados_inicial, dados_opt

def formatar(value, unidade):
    if value > 1000:
        n = 0
    elif value < 3:
        n = 2
    else:
        n = 1
    if pd.notna(value):
        if unidade in [r"^\circ", r"\%"]:
            return "$" + f"{value:.{n}f}" + rf"{unidade}$"
        else:
            return "$" + f"{value:.{n}f}" + rf"\,{unidade}$"
    else:
        return r"$\cdots$"
def tabela_latex(df: pd.DataFrame, unidades: list | None) -> str:
    if unidades is None:
        unidades = [""] * len(df.columns)
        
    def esc(s):
        return s.replace("_", r"\_")

    if any("point" in col for col in df.columns):
        col_list = df.columns
    else:
        col_list = [f"${col}$" for col in df.columns]

    lines = [
        r"\begin{tabular}{l|" + "c" * len(df.columns) + "}",
        " & " + " & ".join(col_list) +
        r" \\",
        r"\hline",
    ]
    
    if any("point" in col for col in df.columns):
        for (index, row), unidade in zip(list(df.iterrows()), unidades):
            values = [formatar(value, unidade) for value in row]
            lines.append(f"{esc(index)} & " + " & ".join(values) + r" \\")

    else:
        for index, row in df.iterrows():
            values = [formatar(value, unidade) for value, unidade in zip(row, unidades)]
            lines.append(f"{esc(index)} & " + " & ".join(values) + r" \\")


    lines.append(r"\end{tabular}")
    return "\n".join(lines)

def mudar_escala(df: pd.DataFrame) -> None:
    linhas_angulo = df.index.str.contains(r"alpha|phi|sweep", case=False, regex=True)
    linhas_porcentagem = df.index.str.contains(r"frac|excess|_c_w|_b_w|SM_", case=False, regex=True)
    linhas_peso = df.index.str.contains(r"MTOW", case=False, regex=True)
    df.loc[linhas_angulo, :] = np.rad2deg(df.loc[linhas_angulo, :])
    df.loc[linhas_porcentagem, :] = df.loc[linhas_porcentagem, :]*100
    df.loc[linhas_peso, :] = df.loc[linhas_peso, :]/gravity

# 2) Default Aircraft Optimization
Minimizar W0(x)

w.r.t. x = [AR_w, S_w]

x0 = [7.5, 90]

Restrições:

[7, 80] <= [AR_w, S_w] <= [12, 120]

b_w(x) - 30 <= 0

In [ ]:
airplane_name = "fokker100"
fokker100_inicial = criar_airplane_parametros(airplane_name, {"AR_w": 9, "S_w": 100})
analyze(fokker100_inicial)

parametros_iniciais = np.array([9, 100])

parametros_limites = {
    "AR_w": [7, 12],
    "S_w": [80, 120]
}


restricoes_limites = {
    "b_w": [None, 30]
}

parametros_hist = []
W0_hist = []
g_hist = []
cache = {}
result, elapsed, parametros, fokker100_opt, dados_fokker100_inicial, dados_fokker100_opt = otimizar_aviao(airplane_name, parametros_limites, parametros_iniciais, restricoes_limites)

print(result)
print(parametros)

## 1. Fill the following table with the optimization results:

In [ ]:
tabela = []
for airplane in (fokker100_inicial, fokker100_opt):
    AR_w = airplane["inputs"]["AR_w"]
    S_w = airplane["inputs"]["S_w"]
    MTOW = airplane["thrust_matching"]["W0"] / gravity
    b_w = airplane["geometry"]["b_w"]
    tabela.append([AR_w, S_w, MTOW, b_w])
df = pd.DataFrame(
    tabela,
    index=["Starting point", "Optimized point"],
    columns=["AR_w", "S_w", "MTOW", "b_w"]
)
print(df)

LaTeX:

In [ ]:
unidades = ["", r"\text{m}^2", r"\text{kgf}", r"\text{m}"]
latex = tabela_latex(df, unidades)
print(latex)

## 2. What is the relative improvement of the objective function, in percentage, after the optimization?

In [ ]:
W0_inicial = fokker100_inicial["thrust_matching"]["W0"]
W0_opt = fokker100_opt["thrust_matching"]["W0"]
print(f"MTOW inicial: {W0_inicial/gravity:.2f} kgf")
print(f"MTOW otimizado: {W0_opt/gravity:.2f} kgf")
print(f"Redução: {(W0_inicial - W0_opt)/gravity:.2f} kgf ({(W0_inicial - W0_opt)/W0_inicial*100:.2f}%)")

## 3. How many function calls were needed in this optimization?

In [ ]:
print(f"{result.nfev} chamadas")

## 4. Is the optimum constrained? What are the active constraints?
Não, pois b_w < 30

# 3) Team Aircraft Optimization

In [ ]:
airplane_name = "Tomav"
tomav_inicial = standard_airplane(airplane_name)
analyze(tomav_inicial)

parametros_limites = { # parâmetro: (mínimo, máximo)
    "delta_xr_w": (-8, 0),
    "S_w": (250, 400),
    "sweep_w": (np.deg2rad(25), np.deg2rad(45)),
    "AR_w": (6, 10.5),
    "taper_w": (0.17, 1),
    "x_tank_c_w": (0.1, 0.7),
    "c_flap_c_wing": (0.15, 0.33),
    "b_flap_b_wing": (0.40, 0.65),
    "b_slat_b_wing": (0.65, 0.90),
    "c_slat_c_wing": (0.08, 0.17),
}

nomes_parametros = list(parametros_limites.keys())
parametros_iniciais = np.array([tomav_inicial["inputs"][parametro] for parametro in parametros_limites])

restricoes_limites = { # restrição: (mínimo, máximo), None para ignorar
  "deltaS_wlan": (0, None), # 1
  "SM_fwd": (None, 0.30), # 2
  "SM_aft": (0.05, None), # 3
  "frac_nlg_aft": (0.03, None), # 4
  "frac_nlg_fwd": (None, 0.18), # 5
  "alpha_tipback": (np.deg2rad(15), None), # 6
  "alpha_tailstrike": (np.deg2rad(10), None), # 7
  "phi_overturn": (None, np.deg2rad(63)), # 8
  # Restrição 9 já está definida no input b_tank_b_w = 0.95
  "b_w": (52, 62), # 10
  # Restrição 11 já está definida no input y_mlg = 5.5
  "tail_height": (None, 20), # 12
  # Restrições adicionais:
  "CLv": (None, 0.75), # 13
  "tank_excess": (0.01, None), # 14
  "mlg_xcg_margin": (0, None), # 15
  "d_TO": (None, 2900) # 16
}

parametros_hist = []
W0_hist = []
g_hist = []
cache = {}
result, elapsed, parametros, tomav_opt, dados_tomav_inicial, dados_tomav_opt = otimizar_aviao(airplane_name, parametros_limites, parametros_iniciais, restricoes_limites)

print(result)

## 1. Give the optimization problem definition and reasons behind the selection of design variables.
Minimizar W0(x)

w.r.t. x = 10 variáveis:

1. delta_xr_w (altera o x da asa, rodas e motores junto)
2. S_w
3. sweep_w
4. AR_w
5. taper_w
6. x_tank_c_w
7. c_flap_c_wing
8. b_flap_b_wing
9. b_slat_b_wing
10. c_slat_c_wing

x0 = avião final PRJ-22

Restrições:

1. Landing requirement: deltaS_wlan ≥ 0
2. Static stability: SM_fwd ≤ 0.30
3. Static stability: SM_aft ≥ 0.05
4. Nose landing gear weight fraction: frac_nlg_fwd ≤ 0.18
5. Nose landing gear weight fraction: frac_nlg_aft ≥ 0.03
6. Main landing gear position: alpha_tipback ≥ 15 deg
7. Main landing gear position: alpha_tailstrike ≥ 10 deg
8. Main landing gear position: phi_overturn ≤ 63 deg
9. Fuel tank volume: b_tank_b_w ≤ 0.95 (já está definido b_tank_b_w = 0.95)
10. Wingspan limit for ICAO gate constraint: see Tab. 1
  - Wingspan = b_w = 58.86425061104575m -> Code E -> 52m < b_w < 65m
11. Wheel span limit for ICAO gate constraint: see Tab. 1
  - Code E => 9 < 2*y_mlg < 14
  - 9 < 2*5.5 < 14 => OK (não entra como restrição)
12. Heihgt limit for FAA Airplane Design Group: see Tab. 2
- Wingspan = 58.86425061104575m => ADG V
- Tail Height = zt_v - z_lg = 12.507446283245763m - (-5.585m) = 18,09244628m => ADG IV
- Pior: ADG V => Tail Height = zt_v - z_lg < 20m

## 2. Explain if new constraints or objectives were added.
Restrições adicionais:
1.  CLv <= 0.75 (controle lateral)
2.  tank_excess >= 0 (combustível suficiente)
3.  x_mlg >= x_cg_aft (não tombar)
4.  d_TO < 2900 (decolar)

## 3. Generate a table comparing values of design variables, constraints, and objective of the initial and the optimized configurations.

Parâmetros e objetivo:

In [ ]:
tabela = {titulo: {parametro: airplane["inputs"][parametro] for parametro in nomes_parametros} for titulo, airplane in zip(["Starting point", "Optimized point"], (tomav_inicial, tomav_opt))}
tabela["Starting point"]["MTOW"] = dados_tomav_inicial["W0"]
tabela["Optimized point"]["MTOW"] = dados_tomav_opt["W0"]
limites = {"Mínimo": {parametro: limite[0] for parametro, limite in parametros_limites.items()},
           "Máximo": {parametro: limite[1] for parametro, limite in parametros_limites.items()}}
limites["Mínimo"]["MTOW"] = None
limites["Máximo"]["MTOW"] = None
df = pd.DataFrame({**tabela, **limites})

linhas_angulo = df.index.str.contains(r"alpha|phi|sweep", case=False, regex=True)
linhas_porcentagem = df.index.str.contains(r"frac|excess|_c_w|_b_w|SM_", case=False, regex=True)
linhas_peso = df.index.str.contains(r"MTOW", case=False, regex=True)
df.loc[linhas_angulo, :] = np.rad2deg(df.loc[linhas_angulo, :])
df.loc[linhas_porcentagem, :] = df.loc[linhas_porcentagem, :]*100
df.loc[linhas_peso, :] = df.loc[linhas_peso, :]/gravity
print(df)


LaTeX

In [ ]:
unidades = [r"\text{m}", r"\text{m}^2", r"^\circ", "", "", r"\%", r"\%", r"\%", r"\%", r"\%", r"\text{kgf}"]
latex = tabela_latex(df, unidades)
print(latex)

Restrições:

In [ ]:
tabela = {titulo: {restricao: dados[restricao] for restricao in restricoes_limites}for titulo, dados in zip(["Starting point", "Optimized point"], (dados_tomav_inicial, dados_tomav_opt))}

limites = {"Mínimo": {restricao: limite[0] for restricao, limite in restricoes_limites.items()},
           "Máximo": {restricao: limite[1] for restricao, limite in restricoes_limites.items()}}
df = pd.DataFrame({**tabela, **limites})
mudar_escala(df)
print(df)

LaTeX

In [ ]:
unidades = [r"\text{m}^2", r"\%", r"\%", r"\%", r"\%", r"^\circ", r"^\circ", r"^\circ", r"\text{m}", r"\text{m}", "", r"\%", r"\text{m}", r"\text{m}"]
latex = tabela_latex(df, unidades)
print(latex)

## 4. Which optimizer have you used? Why?"
scipy.optimize.minimize com SLSQP, porque usei antes

## 5. What is the relative improvement of the objective function, in percentage, after the optimization?

In [ ]:
W0_inicial = tomav_inicial["thrust_matching"]["W0"]
W0_opt = tomav_opt["thrust_matching"]["W0"]
print(f"MTOW inicial: {W0_inicial/gravity:.2f} kgf")
print(f"MTOW otimizado: {W0_opt/gravity:.2f} kgf")
print(f"Redução: {(W0_inicial - W0_opt)/gravity:.2f} kgf ({(W0_inicial - W0_opt)/W0_inicial*100:.2f}%)")

## 6. How many function calls were needed in this optimization? How long did it take (seconds)?

In [ ]:
print(f"{result.nfev} chamadas")
print(f"{elapsed} segundos")

## 7. Generate charts with the optimization history of design variables, constraints, and objective.

In [ ]:
parametros_data = pd.DataFrame(np.vstack(parametros_hist), columns=nomes_parametros)

parametros_percentuais = ["c_flap_c_wing", "b_flap_b_wing", "c_slat_c_wing", "b_slat_b_wing", "x_tank_c_w"]
parametros_percentuais_data = parametros_data[parametros_percentuais]
parametros_data = parametros_data.drop(columns=parametros_percentuais)
parametros_data["sweep_w"] = np.rad2deg(parametros_data["sweep_w"])
parametros_data["S_w"] = parametros_data["S_w"]/10
parametros_data.columns = ["delta_xr_w (m)", "S_w (m²) / 10 ", "sweep_w (°)", "AR_w", "taper_w"]

nomes_g = []
for restricao, (limite_inferior, limite_superior) in restricoes_limites.items():
    if limite_inferior is not None:
        nomes_g.append(f"{restricao} min")
    if limite_superior is not None:
        nomes_g.append(f"{restricao} max")

g_data = pd.DataFrame(g_hist, columns=nomes_g)
g_data["deltaS_wlan min / 100"] = g_data["deltaS_wlan min"] / 100
g_data.drop(columns=["deltaS_wlan min"], inplace=True)
max_x = len(W0_hist)

plt.figure(figsize=(16, 11), constrained_layout=True)

plt.subplot(221)
sns.lineplot(data=parametros_data, dashes=False, linewidth=2)
plt.ylabel("Parâmetro")
plt.xlim(0, max_x)
plt.legend(loc="upper right")

plt.subplot(222)
sns.lineplot(data=parametros_percentuais_data, dashes=False, linewidth=2)
plt.gca().yaxis.set_major_formatter(PercentFormatter(xmax=1))
plt.ylim(0, 1)
plt.ylabel("Parâmetro")
plt.xlim(0, max_x)
plt.legend(loc="upper right")

plt.subplot(223)
sns.lineplot(x=range(len(W0_hist)), y=W0_hist, dashes=False, linewidth=2)
plt.ylabel("MTOW (kgf)")
plt.xlabel("evaluations")

plt.subplot(224)
sns.lineplot(data=g_data, linewidth=2)
plt.ylabel("Restrição")
plt.xlabel("evaluations")
plt.xlim(0, max_x)
plt.ylim(-1, 5)
plt.legend(loc="upper right")
plt.savefig("Lab2/history.png", dpi=300)

## 8. Is the optimum constrained? What are the active constraints?

Sim, ver onde Optimized Point coincide com Mínimo ou Máximo na tabela de restrições

# 4) Multiobjective Optimization
Minimizar W0(x) e Wf(x)

w.r.t. x = 10 variáveis

x0 = avião final PRJ-22

Restrições: iguais

Use MOGA algorithm from pymoo.

In [ ]:
airplane_name = "Tomav"
n_var = len(parametros_limites)
n_ieq_constr = 0
for limite_inferior, limite_superior in restricoes_limites.values():
    if limite_inferior is not None:
        n_ieq_constr += 1
    if limite_superior is not None:
        n_ieq_constr += 1

lb = np.array([limite[0] for limite in parametros_limites.values()])
ub = np.array([limite[1] for limite in parametros_limites.values()])
nomes_parametros = list(parametros_limites.keys())
x_norm_inicial = normalizar(parametros_iniciais, lb, ub)
dados_inicial = calcular_dados_otimizacao(airplane_name, x_norm_inicial, lb, ub, nomes_parametros)
if dados_inicial is None:
    raise ValueError
W0_inicial = dados_inicial["W0"]
Wf_inicial = dados_inicial["Wf"]

class ProblemaW0WfTomav(ElementwiseProblem):
    def __init__(self):
        # Set general characteristics of the problem
        super().__init__(n_var=n_var, # Number of design variables
                         n_obj=2, # Number of objective functions
                         n_ieq_constr=n_ieq_constr, # Number of inequality constraints
                         xl=np.array([0]*n_var), # Lower bounds of design variables
                         xu=np.array([1]*n_var)) # Upper bound of design variables
    def _evaluate(self, x_norm, out, *args, **kwargs):
        dados = calcular_dados_otimizacao(airplane_name, x_norm, lb, ub, nomes_parametros)
        W0_norm = dados["W0"] / W0_inicial if dados is not None else 1e10
        Wf_norm = dados["Wf"] / Wf_inicial if dados is not None else 1e10
        g = confun(x_norm, airplane_name, lb, ub, nomes_parametros, restricoes_limites)
        out["F"] = np.array([W0_norm, Wf_norm])
        out["G"] = [-k for k in g]
problem = ProblemaW0WfTomav()
algorithm = NSGA2(pop_size=400, eliminate_duplicates=True)
res = minimize(problem,
               algorithm,
               ('n_gen', 200),
               seed=2,
               verbose=True)

## 1. Plot the Pareto Front.

In [ ]:
W0_opt = res.F[:, 0] * W0_inicial / gravity
Wf_opt = res.F[:, 1] * Wf_inicial / gravity
x_opt = [desnormalizar(x_norm, lb, ub) for x_norm in res.X]
plt.figure()
sns.scatterplot(x=W0_opt, y=Wf_opt)
plt.xlabel(r"$W_0$")
plt.ylabel(r"$W_f$")
plt.tight_layout()
plt.ticklabel_format(style="plain", axis="both", useOffset=False)
plt.savefig("Lab2/pareto.png", dpi=300)

## 2. Show the number of generations and individuals per generation.

100 gerações, 1000 indivíduos

## 3. How would you use the results from Sec. 3 to verify the convergence of this multiobjective optimization?

Ponto com melhor W0 deve quase coincidir com a otimização baseada só no W0

## 4. Choose at least three aircraft from distinct regions of the Pareto Front. Draw their planforms and discuss how their differences impact the objective function values.

In [ ]:
airplane1 = 
airplane2 = 
airplane3 =
tomav_inicial

Mostrar figuras

In [ ]:
plt.show()